# Question 4 - Merge protected areas from WDPA database

In [1]:
# Import packages
import pandas as pd
import geopandas as gpd
import zipfile
import os

## Load GBIF data

In [2]:
# Load GBIF data from Google Drive
url = 'https://drive.google.com/file/d/1dJDqfHaZCnPdiBqgHah64KGEHiSNiCw0/view?usp=sharing'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
df_gbif_as = pd.read_csv(path)

/var/folders/v8/1_2rlmrd0wl5w5t5n60h8s440000gn/T/ipykernel_72005/3102567984.py:4: DtypeWarning: Columns (0: eventID, 1: http://unknown.org/language, 2: footprintWKT, 3: http://unknown.org/modified, 4: originalNameUsage, 5: nameAccordingTo, 6: identificationRemarks) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gbif_as = pd.read_csv(path)


In [3]:
df_gbif_as.head()

,key,datasetKey,publishingOrgKey,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,crawlId,...,dynamicProperties,informationWithheld,networkKeys,elevation,higherGeography,higherGeographyID,nomenclaturalCode,bibliographicCitation,fieldNumber,distanceFromCentroidInMeters
0,4148782383,9fd6c3da-88f7-4df7-9a3f-3f3f9c703998,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,e44d0fd7-0edf-477f-aa82-50a81836ab46,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,FR,DWC_ARCHIVE,2026-03-04T06:26:31.021+00:00,2026-03-04T06:34:03.803+00:00,365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4148782475,9fd6c3da-88f7-4df7-9a3f-3f3f9c703998,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,e44d0fd7-0edf-477f-aa82-50a81836ab46,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,FR,DWC_ARCHIVE,2026-03-04T06:26:31.021+00:00,2026-03-04T06:34:12.055+00:00,365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4949483878,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:05.699+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4953954153,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:47.875+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4977609431,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:40.764+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_gbif_as.info()

<class 'pandas.DataFrame'>
RangeIndex: 17866 entries, 0 to 17865
Columns: 117 entries, key to distanceFromCentroidInMeters
dtypes: bool(2), float64(8), int64(17), str(90)
memory usage: 84.2 MB


In [5]:
list(df_gbif_as.columns)

['key',
 'datasetKey',
 'publishingOrgKey',
 'installationKey',
 'hostingOrganizationKey',
 'publishingCountry',
 'protocol',
 'lastCrawled',
 'lastParsed',
 'crawlId',
 'extensions',
 'basisOfRecord',
 'occurrenceStatus',
 'classifications',
 'taxonKey',
 'kingdomKey',
 'phylumKey',
 'classKey',
 'orderKey',
 'familyKey',
 'genusKey',
 'speciesKey',
 'acceptedTaxonKey',
 'scientificName',
 'scientificNameAuthorship',
 'acceptedScientificName',
 'kingdom',
 'phylum',
 'order',
 'family',
 'genus',
 'species',
 'genericName',
 'specificEpithet',
 'infraspecificEpithet',
 'taxonRank',
 'taxonomicStatus',
 'decimalLatitude',
 'decimalLongitude',
 'coordinateUncertaintyInMeters',
 'continent',
 'gadm',
 'year',
 'month',
 'day',
 'eventDate',
 'startDayOfYear',
 'endDayOfYear',
 'issues',
 'lastInterpreted',
 'license',
 'isSequenced',
 'identifiers',
 'media',
 'facts',
 'relations',
 'isInCluster',
 'datasetID',
 'recordedBy',
 'identifiedBy',
 'dnaSequenceID',
 'geodeticDatum',
 'class'

In [6]:
# Check projection
df_gbif_as['geodeticDatum'].unique()

<ArrowStringArray>
['WGS84', nan]
Length: 2, dtype: str

In [7]:
# Convert to GeoDataFrame
gdf_gbif = gpd.GeoDataFrame(
    df_gbif_as,
    geometry=gpd.points_from_xy(
        df_gbif_as["decimalLongitude"],
        df_gbif_as["decimalLatitude"]
    ),
    crs="EPSG:4326"  # WGS84
)

display(gdf_gbif.head())
print(gdf_gbif.crs)
print(gdf_gbif.shape)

,key,datasetKey,publishingOrgKey,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,crawlId,...,informationWithheld,networkKeys,elevation,higherGeography,higherGeographyID,nomenclaturalCode,bibliographicCitation,fieldNumber,distanceFromCentroidInMeters,geometry
0,4148782383,9fd6c3da-88f7-4df7-9a3f-3f3f9c703998,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,e44d0fd7-0edf-477f-aa82-50a81836ab46,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,FR,DWC_ARCHIVE,2026-03-04T06:26:31.021+00:00,2026-03-04T06:34:03.803+00:00,365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (9.35013 48.6616)
1,4148782475,9fd6c3da-88f7-4df7-9a3f-3f3f9c703998,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,e44d0fd7-0edf-477f-aa82-50a81836ab46,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,FR,DWC_ARCHIVE,2026-03-04T06:26:31.021+00:00,2026-03-04T06:34:12.055+00:00,365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (8.51561 49.24991)
2,4949483878,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:05.699+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (7.46331 49.45567)
3,4953954153,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:47.875+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (8.26295 49.35314)
4,4977609431,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:40.764+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (8.04916 49.38645)


EPSG:4326
(17866, 118)


In [8]:
# Remove rows with missing coordinates
gdf_gbif = gdf_gbif.dropna(subset=["decimalLatitude", "decimalLongitude"])
print(gdf_gbif.shape)

(17863, 118)


In [9]:
# Check if "key" column is a unique identifier
print(gdf_gbif["key"].nunique())
print(gdf_gbif["key"].is_unique)

17863
True


## Load WDPA data

In [10]:
# Your zip files (adjust path)
zip_files = [
    "/Users/Wanja/Documents/repos/nabu-asian-hornet-project-research/WDPA_WDOECM_Apr2026_Public_DEU_shp/WDPA_WDOECM_Apr2026_Public_DEU_shp_0.zip",
    "/Users/Wanja/Documents/repos/nabu-asian-hornet-project-research/WDPA_WDOECM_Apr2026_Public_DEU_shp/WDPA_WDOECM_Apr2026_Public_DEU_shp_1.zip",
    "/Users/Wanja/Documents/repos/nabu-asian-hornet-project-research/WDPA_WDOECM_Apr2026_Public_DEU_shp/WDPA_WDOECM_Apr2026_Public_DEU_shp_2.zip"
]

extract_root = "temp_wdpa"
os.makedirs(extract_root, exist_ok=True)

gdfs = []

for i, zip_path in enumerate(zip_files):
    extract_dir = os.path.join(extract_root, f"part_{i}")
    os.makedirs(extract_dir, exist_ok=True)

    # Extract
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)

    # Find shapefiles
    shp_files = [
        os.path.join(extract_dir, f)
        for f in os.listdir(extract_dir)
        if f.endswith(".shp")
    ]

    # Load all wdpa shapefiles in this zip
    for shp in shp_files:
        gdf = gpd.read_file(shp)
        gdfs.append(gdf)

# Combine all wdpa shapefiles into one GeoDataFrame
gdf_wdpa_all = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

print(gdf_wdpa_all.shape)
print(gdf_wdpa_all.crs)
print(gdf_wdpa_all.columns)
display(gdf_wdpa_all.head())

(23609, 34)
EPSG:4326
Index(['SITE_ID', 'SITE_PID', 'SITE_TYPE', 'NAME_ENG', 'NAME', 'DESIG',
       'DESIG_ENG', 'DESIG_TYPE', 'IUCN_CAT', 'INT_CRIT', 'REALM',
       'REP_M_AREA', 'GIS_M_AREA', 'REP_AREA', 'GIS_AREA', 'NO_TAKE',
       'NO_TK_AREA', 'STATUS', 'STATUS_YR', 'GOV_TYPE', 'GOVSUBTYPE',
       'OWN_TYPE', 'OWNSUBTYPE', 'MANG_AUTH', 'MANG_PLAN', 'VERIF',
       'METADATAID', 'PRNT_ISO3', 'ISO3', 'SUPP_INFO', 'CONS_OBJ',
       'INLND_WTRS', 'OECM_ASMT', 'geometry'],
      dtype='str')


,SITE_ID,SITE_PID,SITE_TYPE,NAME_ENG,NAME,DESIG,DESIG_ENG,DESIG_TYPE,IUCN_CAT,INT_CRIT,...,MANG_PLAN,VERIF,METADATAID,PRNT_ISO3,ISO3,SUPP_INFO,CONS_OBJ,INLND_WTRS,OECM_ASMT,geometry
0,667,667,PA,Bayerischer Wald,Bayerischer Wald,Nationalpark,National Park,National,II,Not Applicable,...,Not Reported,State Verified,2013,DEU,DEU,Not Applicable,Not Applicable,Not Reported,Not Applicable,"MULTIPOLYGON (((13.50728 48.89217, 13.50716 48..."
1,668,668,PA,Berchtesgaden,Berchtesgaden,Nationalpark,National Park,National,II,Not Applicable,...,Not Reported,State Verified,2013,DEU,DEU,Not Applicable,Not Applicable,Not Reported,Not Applicable,"POLYGON ((12.82676 47.64227, 12.82757 47.64195..."
2,1538,1538,PA,Oberharz,Oberharz,Naturschutzgebiet,Nature Reserve,National,IV,Not Applicable,...,Not Reported,State Verified,2013,DEU,DEU,Not Applicable,Not Applicable,Not Reported,Not Applicable,"MULTIPOLYGON (((10.58678 51.7413, 10.58732 51...."
3,1542,1542,PA,Federsee,Federsee,Naturschutzgebiet,Nature Reserve,National,IV,Not Applicable,...,Not Reported,State Verified,2013,DEU,DEU,Not Applicable,Not Applicable,Not Reported,Not Applicable,"POLYGON ((9.63773 48.10648, 9.63848 48.10624, ..."
4,4380,4380,PA,Rantumbecken,Rantumbecken,Naturschutzgebiet,Nature Reserve,National,IV,Not Applicable,...,Not Reported,State Verified,2013,DEU,DEU,Not Applicable,Not Applicable,Not Reported,Not Applicable,"POLYGON ((8.33829 54.87973, 8.33775 54.87912, ..."


In [11]:
# Check geometry types
geom_counts = gdf_wdpa_all.geometry.geom_type.value_counts()
print(geom_counts)

Polygon         15767
MultiPolygon     7829
MultiPoint         13
Name: count, dtype: int64


## Intersect GBIF & WDPA data

Both datasets use the CRS WGS84, no reprojection necessary. 
The wdpa data contains mostly polygons and only 13 multipoints. A spatial join looking for occurrences within polygons and ignoring the 13 points is therefore appropriate.
The "key" column in the gbif data can act as a unique identifier for merging data back to the gbif. 

In [12]:
# Spatial join: find which GBIF occurrences fall within WDPA polygons
gdf_joined = gdf_gbif.sjoin(
    gdf_wdpa_all[["geometry", "DESIG_ENG", "IUCN_CAT"]],  # only need geometry
    how="left",
    predicate="within" # within excludes points exactly on the boundary
)
print(gdf_joined.shape)

(19422, 121)


In [13]:
# Aggregate results by "key" to determine if each occurrence is inside a protected area and to collect designations/categories
# This gets rid of duplicate occurrences that fall within multiple protected areas, but we can still capture all designations/categories in lists
agg = (
    gdf_joined
    .groupby("key")
    .agg({
        "index_right": lambda x: x.notna().any(),
        "DESIG_ENG": lambda x: list(x.dropna().unique()),
        "IUCN_CAT": lambda x: list(x.dropna().unique())
    })
    .rename(columns={"index_right": "inside_protected_area"})
    .reset_index()
)

# Join back to original gbif data (preserves all columns)
gdf_result = gdf_gbif.merge(
    agg,
    on="key",
    how="left"
)

# Fill NaN (points outside protected areas) with False
gdf_result["inside_protected_area"] = gdf_result["inside_protected_area"].fillna(False)

# Check: no row loss
assert gdf_result.shape[0] == gdf_gbif.shape[0]
print(gdf_result["inside_protected_area"].value_counts())
print(gdf_result.shape)
display(gdf_result.head())

inside_protected_area
False    14473
True      3390
Name: count, dtype: int64
(17863, 121)


,key,datasetKey,publishingOrgKey,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,crawlId,...,higherGeography,higherGeographyID,nomenclaturalCode,bibliographicCitation,fieldNumber,distanceFromCentroidInMeters,geometry,inside_protected_area,DESIG_ENG,IUCN_CAT
0,4148782383,9fd6c3da-88f7-4df7-9a3f-3f3f9c703998,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,e44d0fd7-0edf-477f-aa82-50a81836ab46,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,FR,DWC_ARCHIVE,2026-03-04T06:26:31.021+00:00,2026-03-04T06:34:03.803+00:00,365,...,NaN,NaN,NaN,NaN,NaN,NaN,POINT (9.35013 48.6616),False,[],[]
1,4148782475,9fd6c3da-88f7-4df7-9a3f-3f3f9c703998,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,e44d0fd7-0edf-477f-aa82-50a81836ab46,1928bdf0-f5d2-11dc-8c12-b8a03c50a862,FR,DWC_ARCHIVE,2026-03-04T06:26:31.021+00:00,2026-03-04T06:34:12.055+00:00,365,...,NaN,NaN,NaN,NaN,NaN,NaN,POINT (8.51561 49.24991),False,[],[]
2,4949483878,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:05.699+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,POINT (7.46331 49.45567),False,[],[]
3,4953954153,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:47.875+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,POINT (8.26295 49.35314),False,[],[]
4,4977609431,aa6c5ee6-d4d7-4a65-a04f-379cffbf4842,2754e9c0-0e43-4f65-968a-6f16b9c378ce,dcceb601-2fb0-49dc-9cd2-7c00056f2b2c,2754e9c0-0e43-4f65-968a-6f16b9c378ce,DE,BIOCASE,2026-03-20T00:46:08.174+00:00,2026-03-20T00:58:40.764+00:00,384,...,NaN,NaN,NaN,NaN,NaN,NaN,POINT (8.04916 49.38645),True,"[biosphere reserve, biosphere reserve - buffer...",[Not Assigned]


## TO DO 

- check if gbif data is complete and clean 

- area weighted chi-square test
    - load germany boundaries
    - calculate area of protected areas and non-protected areas
    - do chi-square on occurrences per area
- monte carlo spatial null
    - generate random occurrences
    - compare?

- check for differences between protected area types
    - adjust spatial join and counting of occurrences per protected area
    - plot or chi-square again

- what about bias from people being allowed to walk there or not?

- visualizations

## References

UNEP-WCMC and IUCN (2026), Protected Planet: The World Database on Protected Areas (WDPA) and World Database on Other Effective Area-based Conservation Measures (WD-OECM) [Online], April 2026, Cambridge, UK: UNEP-WCMC and IUCN. Available at: www.protectedplanet.net.